# 🎮 LoRA Fine-tuning v2 (Game-optimized / KcELECTRA)

**보정된 UnSmile + 수집된 게임 음성채팅 데이터**로 학습합니다.

- **모델**: `beomi/KcELECTRA-base-v2022`
- **메트릭**: `abuse_recall`
- **데이터**: UnSmile 보정 10,490 + 수집 519 = **11,009건**

In [ ]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
OUTPUT_DIR = "./output/lora_game_kcelectra_v2"  # v2 폴더 내 output 폴더에 저장
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS, BATCH_SIZE, LEARNING_RATE = 10, 32, 2e-4
MAX_LENGTH = 128
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# 데이터 로드 및 병합 (v2: 보정 + 수집)
unsmile_train = pd.read_csv("../../3_UnSmile_Correction/unsmile_train_corrected.tsv", sep='\t')
valid_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_valid_corrected.tsv", sep='\t')
collected_df = pd.read_csv("../../1_Data_Labeling_STT/keywords_unsmile_format.tsv", sep='\t')

train_df = pd.concat([unsmile_train, collected_df], ignore_index=True)
print(f"✅ Train: {len(train_df)}건 (보정 {len(unsmile_train)} + 수집 {len(collected_df)})")
print(f"Valid: {len(valid_df)}건")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(examples['문장'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [[float(examples[col][i]) for col in LABEL_NAMES] for i in range(len(examples['문장']))]
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification")
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=["query", "key", "value"], bias="none")
model = get_peft_model(base_model, peft_config).to(DEVICE)
model.print_trainable_parameters()

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    _, abuse_r, abuse_f1, _ = precision_recall_fscore_support(labels_int[:,8], preds[:,8], average='binary', zero_division=0)
    _, clean_r, clean_f1, _ = precision_recall_fscore_support(labels_int[:,9], preds[:,9], average='binary', zero_division=0)
    return {'lrap': label_ranking_average_precision_score(labels, predictions), 'abuse_recall': abuse_r, 'abuse_f1': abuse_f1, 'clean_recall': clean_r, 'clean_f1': clean_f1}

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="abuse_recall", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", fp16=True
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
                  tokenizer=tokenizer, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

In [ ]:
print("🚀 v2 학습 시작...")
trainer.train()
print("학습 완료!")

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")
print("✅ v2 모델 저장 완료!")

eval_results = trainer.evaluate()
for k, v in eval_results.items(): print(f"  {k}: {v:.4f}")